In [0]:
# Databricks notebook source
# COMMAND ----------
import os
import json
import base64
import requests
from datetime import datetime, timezone, timedelta

# 1. WIDGET CREDENTIALS
# 1. DATABRICKS SECRETS CREDENTIALS
CLIENT_ID = dbutils.secrets.get(scope="ebay_api", key="client_id")
CLIENT_SECRET = dbutils.secrets.get(scope="ebay_api", key="client_secret")

# Fallback direct path definition (Resolves NameError: RAW_INC_DIR)
REPO_BASE = "/Workspace/Users/ammaraa5688@gmail.com/Online-retail-data-pipeline"
RAW_INC_DIR = os.path.join(REPO_BASE, "data", "raw", "incremental_load")
WATERMARK_FILE = os.path.join(RAW_INC_DIR, ".watermark.txt")

# Ensure target landing directory exists
os.makedirs(RAW_INC_DIR, exist_ok=True)

# 2. TARGET SUBCATEGORIES TO PULL
TARGET_SUBCATEGORIES = {
    "43304": "cell phones and accessories-cell phones & smartphone parts",
    "178893": "cell phones and accessories-smartwatches",
    "175672": "coin & paper money-bullion",
    "3412": "coin & paper money-us paper money",
    "26429": "eBay Motor-Other Vehicles & Trailers",
    "4196": "jewelery and watches-fine jewelery",
    "10968": "jewelry and watches-fashion jewelery",
    "155101": "jewelry and watches-kids jewelry",
    "15841": "real estate-commercial real estate",
    "15840": "real estate-land read estate",
    "12605": "real estate-residential real estate",
    "15273": "sporting goods- fitness, running and yoga",
    "16034": "sporting goods-camping and hiking",
    "1513": "sporting goods-golf"
}

# 3. HELPER FUNCTIONS
def get_oauth_token():
    url = "https://api.ebay.com/identity/v1/oauth2/token"
    creds = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {creds}"
    }
    body = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope"
    }
    res = requests.post(url, headers=headers, data=body, timeout=15)
    res.raise_for_status()
    return res.json()["access_token"]

def get_watermark():
    if os.path.exists(WATERMARK_FILE):
        with open(WATERMARK_FILE, "r", encoding="utf-8") as f:
            ts = f.read().strip()
            if ts:
                return ts
    return (datetime.now(timezone.utc) - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

def save_watermark(ts_str):
    with open(WATERMARK_FILE, "w", encoding="utf-8") as f:
        f.write(ts_str)

def fetch_subcategory_incremental(token, category_id, watermark, limit=4):
    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    params = {
        "category_ids": category_id,
        "sort": "newlyListed",
        "filter": f"itemCreationDate:[{watermark}..]",
        "limit": limit
    }
    res = requests.get(url, headers=headers, params=params, timeout=15)
    if res.status_code == 200:
        return res.json().get("itemSummaries", [])
    return []

# 4. EXECUTE LIVE INGESTION
if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Missing eBay credentials! Fill the widget text boxes above.")

token = get_oauth_token()
batch_time = datetime.now(timezone.utc)
batch_id = f"batch_{batch_time.strftime('%Y%m%d_%H%M%S')}"
current_watermark = get_watermark()
print(f"Fetching listings created after watermark: {current_watermark}")

bronze_records = []
latest_seen_ts = current_watermark

for cat_id, subcat_name in TARGET_SUBCATEGORIES.items():
    items = fetch_subcategory_incremental(token, cat_id, current_watermark, limit=4)
    for item in items:
        creation = item.get("itemCreationDate")
        if creation and creation > latest_seen_ts:
            latest_seen_ts = creation
        
        bronze_records.append({
            "load_type": "incremental",
            "batch_id": batch_id,
            "ingestion_timestamp": batch_time.isoformat(),
            "subcategory_partition": subcat_name,
            "raw_payload": {
                "itemId": item.get("itemId"),
                "title": item.get("title"),
                "categories": item.get("categories"),
                "price": item.get("price"),
                "seller": item.get("seller", {}).get("username"),
                "condition": item.get("condition"),
                "itemCreationDate": item.get("itemCreationDate")
            }
        })

if bronze_records:
    batch_filename = f"ebay_incremental_{batch_time.strftime('%Y%m%d_%H%M%S')}.json"
    target_path = os.path.join(RAW_INC_DIR, batch_filename)
    with open(target_path, "w", encoding="utf-8") as f:
        json.dump(bronze_records, f, indent=2)
    save_watermark(latest_seen_ts)
    print(f"Successfully landed {len(bronze_records)} records into {target_path}")
else:
    print("No new listings past watermark.")